In [ ]:
# -*- coding: utf-8 -*-
"""
PyTorch GPU thesis experiment (stock-only version, 25 self-features)
----------------------------------------------------
研究目标：
1. A股股票池
2. 仅保留 direct multi-step: 直接预测 [t+1, t+2, t+3]
3. 模型比较：
   - BaseLSTM
   - FeatureAttention_LSTM
   - TemporalAttention_LSTM
   - DualAttention_LSTM
4. 输入特征：
   - 仅使用个股自身技术特征
   - 个股自身特征约 25 个
5. GPU 支持
6. 中文绘图支持
7. 新浪优先抓取数据
8. 训练阶段增加模型收敛速度统计

本版本修改重点：
- 完全移除板块指数与大盘指数特征
- 仅保留个股自身 25 个技术特征
- 其余训练、评估、保存流程尽量保持原始结构一致
"""

from __future__ import annotations

import copy
import json
import random
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Any, Tuple, Optional, Callable

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    import akshare as ak
except Exception:
    ak = None

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Global settings
# ============================================================

SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT_DIR = Path("thesis_25stocks_lstm_attention_stock_only_results")
ROOT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = ROOT_DIR / "data_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_STOCK_DIR = CACHE_DIR / "stocks"
CACHE_META_DIR = CACHE_DIR / "meta"

for _p in [CACHE_STOCK_DIR, CACHE_META_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

START_DATE = "20160101"
END_DATE = "20250701"

LOOKBACK = 90
FORECAST_HORIZONS = 3
TEST_RATIO = 0.20
VAL_RATIO_WITHIN_TRAIN = 0.20
MIN_EFFECTIVE_ROWS = LOOKBACK + FORECAST_HORIZONS + 40

ARCH_NAMES = [
    "BaseLSTM",
    "FeatureAttention_LSTM",
    "TemporalAttention_LSTM",
    "DualAttention_LSTM",
]

# 保留你的股票池结构，其他流程不变
STOCK_POOL = [
    {"symbol": "600036", "name": "招商银行", "sector": "银行"},
    {"symbol": "002594", "name": "比亚迪", "sector": "新能源汽车"},
    {"symbol": "300750", "name": "宁德时代", "sector": "动力电池"},
    {"symbol": "601633", "name": "长城汽车", "sector": "汽车"},
    {"symbol": "000625", "name": "长安汽车", "sector": "汽车"},
    {"symbol": "000338", "name": "潍柴动力", "sector": "工程机械"},
    {"symbol": "601668", "name": "中国建筑", "sector": "建筑"},
    {"symbol": "600276", "name": "恒瑞医药", "sector": "医药"},
    {"symbol": "000538", "name": "云南白药", "sector": "医药"},
    {"symbol": "300760", "name": "迈瑞医疗", "sector": "医疗器械"},
    {"symbol": "600309", "name": "万华化学", "sector": "化工"},
    {"symbol": "000651", "name": "格力电器", "sector": "家电"},
    {"symbol": "600690", "name": "海尔智家", "sector": "家电"},
    {"symbol": "002415", "name": "海康威视", "sector": "电子"},
    {"symbol": "000725", "name": "京东方A", "sector": "面板"},
    {"symbol": "002475", "name": "立讯精密", "sector": "消费电子"},
    {"symbol": "601899", "name": "紫金矿业", "sector": "有色金属"},
]



PAPER_STOCK_SYMBOL = "600519"

# 仅保留个股自身特征，约 25 个
STOCK_CORE_FEATURES = [
    "close",              # 1
    "open",               # 2
    "high",               # 3
    "low",                # 4
    "volume",             # 5
    "amount",             # 6
    "turnover",           # 7
    "hl_spread",          # 8
    "oc_spread",          # 9
    "return_1",           # 10
    "return_2",           # 11
    "return_5",           # 12
    "log_return_1",       # 13
    "ma_5",               # 14
    "ma_10",              # 15
    "ma_20",              # 16
    "close_ma5_ratio",    # 17
    "close_ma10_ratio",   # 18
    "close_ma20_ratio",   # 19
    "rolling_std_5",      # 20
    "rolling_std_10",     # 21
    "momentum_3",         # 22
    "momentum_5",         # 23
    "rsi_14",             # 24
    "macd_diff",          # 25
]


# ============================================================
# Basic setup
# ============================================================

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def setup_matplotlib_chinese():
    candidate_fonts = [
        "SimHei",
        "Microsoft YaHei",
        "STHeiti",
        "PingFang SC",
        "Noto Sans CJK SC",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    matplotlib.rcParams["font.sans-serif"] = candidate_fonts
    matplotlib.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.figsize"] = (12, 5)
    plt.rcParams["savefig.dpi"] = 180


set_seed(SEED)
setup_matplotlib_chinese()


# ============================================================
# Utilities
# ============================================================

def check_dependencies():
    missing = []
    if ak is None:
        missing.append("akshare")
    if missing:
        raise ImportError(
            f"Missing packages: {missing}\n"
            "Please install:\n"
            "pip install akshare pandas numpy matplotlib scikit-learn torch joblib"
        )


def safe_mkdir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MSE": float(mse),
        "RMSE": float(np.sqrt(mse)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_prediction(output: Any) -> torch.Tensor:
    return output[0] if isinstance(output, tuple) else output


def random_sleep(low: float = 0.8, high: float = 1.8, verbose: bool = False):
    sec = random.uniform(low, high)
    if verbose:
        print(f"    [Sleep] 等待 {sec:.2f}s")
    time.sleep(sec)


def sanitize_filename(text: str) -> str:
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ']
    out = text
    for ch in bad:
        out = out.replace(ch, "_")
    return out


def save_df_cache(df: pd.DataFrame, path: Path):
    safe_mkdir(path.parent)
    df.to_pickle(path)


def load_df_cache(path: Path) -> Optional[pd.DataFrame]:
    if path.exists():
        try:
            return pd.read_pickle(path)
        except Exception:
            return None
    return None


def with_retry(
    func: Callable[[], Any],
    max_retries: int = 5,
    sleep_min: float = 1.0,
    sleep_max: float = 3.0,
    desc: str = "请求"
):
    last_err = None
    for i in range(1, max_retries + 1):
        try:
            result = func()
            return result
        except Exception as e:
            last_err = e
            print(f"    [Retry-{i}/{max_retries}] {desc} 失败: {repr(e)}")
            if i < max_retries:
                wait_sec = random.uniform(sleep_min, sleep_max)
                print(f"    [Retry-{i}/{max_retries}] {desc} {wait_sec:.2f}s 后重试...")
                time.sleep(wait_sec)
    raise RuntimeError(f"{desc} 最终失败，已重试 {max_retries} 次。原始错误: {repr(last_err)}")


def to_market_prefixed_symbol(symbol: str) -> str:
    if symbol.startswith(("600", "601", "603", "605", "688", "689")):
        return f"sh{symbol}"
    return f"sz{symbol}"


# ============================================================
# Pure pandas indicators
# ============================================================

def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / (avg_loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(
    close: pd.Series,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9
) -> Tuple[pd.Series, pd.Series, pd.Series]:
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    macd_diff = macd - macd_signal
    return macd, macd_signal, macd_diff


# ============================================================
# Data standardization
# ============================================================

def standardize_stock_df(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    df = df.rename(columns={
        "日期": "trade_date",
        "date": "trade_date",
        "开盘": "open",
        "open": "open",
        "最高": "high",
        "high": "high",
        "最低": "low",
        "low": "low",
        "收盘": "close",
        "close": "close",
        "成交量": "volume",
        "volume": "volume",
        "成交额": "amount",
        "amount": "amount",
        "换手率": "turnover",
        "turnover": "turnover",
    })
    required = ["trade_date", "open", "high", "low", "close", "volume", "amount"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{symbol} 个股数据缺少必要列: {missing}")

    if "turnover" not in df.columns:
        df["turnover"] = np.nan

    keep_cols = ["trade_date", "open", "high", "low", "close", "volume", "amount", "turnover"]
    out = df[keep_cols].copy()
    out["trade_date"] = pd.to_datetime(out["trade_date"])
    for col in ["open", "high", "low", "close", "volume", "amount", "turnover"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.sort_values("trade_date").dropna(subset=["trade_date", "close"]).reset_index(drop=True)
    return out


# ============================================================
# Data fetching with cache + retry (Sina first)
# ============================================================

def _try_stock_daily_sina(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    market_symbol = to_market_prefixed_symbol(symbol)
    df = ak.stock_zh_a_daily(
        symbol=market_symbol,
        start_date=start_date,
        end_date=end_date,
        adjust=""
    )
    if df is None or len(df) == 0:
        raise RuntimeError(f"stock_zh_a_daily 返回空: {symbol}")

    if isinstance(df.index, pd.DatetimeIndex) and "date" not in df.columns:
        df = df.reset_index().rename(columns={df.index.name or "index": "date"})

    df = standardize_stock_df(df, symbol)
    df = df[
        (df["trade_date"] >= pd.to_datetime(start_date)) &
        (df["trade_date"] <= pd.to_datetime(end_date))
    ].copy().reset_index(drop=True)
    return df


def _try_stock_hist_em(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    df = ak.stock_zh_a_hist(
        symbol=symbol,
        period="daily",
        start_date=start_date,
        end_date=end_date,
        adjust=""
    )
    if df is None or len(df) == 0:
        raise RuntimeError(f"stock_zh_a_hist 返回空: {symbol}")
    return standardize_stock_df(df, symbol)


def fetch_single_stock_data(
    symbol: str,
    start_date: str = START_DATE,
    end_date: str = END_DATE,
    use_cache: bool = True
) -> pd.DataFrame:
    check_dependencies()
    cache_path = CACHE_STOCK_DIR / f"{symbol}_{start_date}_{end_date}.pkl"

    if use_cache:
        cached = load_df_cache(cache_path)
        if cached is not None and len(cached) > 0:
            print(f"  [Cache] 个股 {symbol}")
            return cached.copy()

    print(f"  [Fetch] 抓取个股(新浪优先): {symbol}")

    fetch_methods = [
        ("stock_zh_a_daily[Sina-first]", lambda: _try_stock_daily_sina(symbol, start_date, end_date)),
        ("stock_zh_a_hist[EM-fallback]", lambda: _try_stock_hist_em(symbol, start_date, end_date)),
    ]

    errors = []
    for method_name, method in fetch_methods:
        try:
            print(f"    [Try] 使用 {method_name} 获取个股 {symbol}")
            df = with_retry(
                method,
                max_retries=4,
                sleep_min=1.0,
                sleep_max=2.5,
                desc=f"{symbol} via {method_name}"
            )

            if df is None or len(df) == 0:
                raise RuntimeError(f"{method_name} 成功调用但结果为空: {symbol}")

            print(f"    [OK] 个股 {symbol} 使用 {method_name} 成功, rows={len(df)}")
            save_df_cache(df, cache_path)
            return df.copy()

        except Exception as e:
            errors.append(f"{method_name}: {repr(e)}")
            print(f"    [Fallback] {method_name} 失败，继续尝试下一个接口...")

    raise RuntimeError(
        f"Failed to fetch stock data for {symbol}. All methods failed:\n" +
        "\n".join(errors)
    )


# ============================================================
# Feature engineering (stock-only, 25 features)
# ============================================================

def create_stock_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df[["trade_date", "open", "high", "low", "close", "volume", "amount", "turnover"]].copy()
    eps = 1e-8

    # 价差类
    out["hl_spread"] = (out["high"] - out["low"]) / (out["close"] + eps)
    out["oc_spread"] = (out["close"] - out["open"]) / (out["open"] + eps)

    # 收益率类
    out["return_1"] = out["close"].pct_change(1)
    out["return_2"] = out["close"].pct_change(2)
    out["return_5"] = out["close"].pct_change(5)
    out["log_return_1"] = np.log((out["close"] + eps) / (out["close"].shift(1) + eps))

    # 均线类
    out["ma_5"] = out["close"].rolling(5).mean()
    out["ma_10"] = out["close"].rolling(10).mean()
    out["ma_20"] = out["close"].rolling(20).mean()

    out["close_ma5_ratio"] = out["close"] / (out["ma_5"] + eps)
    out["close_ma10_ratio"] = out["close"] / (out["ma_10"] + eps)
    out["close_ma20_ratio"] = out["close"] / (out["ma_20"] + eps)

    # 波动率类
    out["rolling_std_5"] = out["return_1"].rolling(5).std()
    out["rolling_std_10"] = out["return_1"].rolling(10).std()

    # 动量类
    out["momentum_3"] = out["close"] / (out["close"].shift(3) + eps) - 1.0
    out["momentum_5"] = out["close"] / (out["close"].shift(5) + eps) - 1.0

    # RSI / MACD
    out["rsi_14"] = compute_rsi(out["close"], window=14)
    macd, macd_signal, macd_diff = compute_macd(out["close"])
    out["macd"] = macd
    out["macd_signal"] = macd_signal
    out["macd_diff"] = macd_diff

    # 只保留 25 个核心特征
    keep_cols = ["trade_date"] + STOCK_CORE_FEATURES
    out = out[keep_cols].copy()

    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


def prepare_feature_df(
    stock_info: Dict[str, str],
    start_date: str = START_DATE,
    end_date: str = END_DATE
) -> pd.DataFrame:
    symbol = stock_info["symbol"]

    print(f"  [Stage] 抓取个股原始数据 -> {symbol}")
    stock_raw = fetch_single_stock_data(symbol, start_date, end_date)
    feature_df = create_stock_features(stock_raw)

    feature_cols = [c for c in feature_df.columns if c != "trade_date"]
    print(f"  [Info] 当前总特征数: {len(feature_cols)}")
    print(f"  [Info] 特征列表: {feature_cols}")

    return feature_df


# ============================================================
# Sequence builders
# ============================================================

def build_multi_horizon_dataset(
    feature_df: pd.DataFrame,
    lookback: int = LOOKBACK,
    horizons: int = FORECAST_HORIZONS
):
    if len(feature_df) < lookback + horizons:
        raise ValueError("Not enough rows to build multi-horizon dataset.")

    feature_cols = [c for c in feature_df.columns if c != "trade_date"]
    values = feature_df[feature_cols].astype(float).values
    closes = feature_df["close"].astype(float).values
    dates = feature_df["trade_date"].values

    X_list, Y_list, D_list, origin_indices = [], [], [], []

    for end_idx in range(lookback - 1, len(feature_df) - horizons):
        start_idx = end_idx - lookback + 1
        X = values[start_idx:end_idx + 1]
        Y = closes[end_idx + 1:end_idx + horizons + 1]
        D = dates[end_idx + 1:end_idx + horizons + 1]

        X_list.append(X)
        Y_list.append(Y)
        D_list.append(D)
        origin_indices.append(end_idx)

    X = np.asarray(X_list, dtype=np.float32)
    Y = np.asarray(Y_list, dtype=np.float32)
    D = np.asarray(D_list)
    origin_indices = np.asarray(origin_indices, dtype=np.int64)

    return X, Y, D, origin_indices, feature_cols


def chronological_split_indices(n: int, test_ratio: float = TEST_RATIO):
    split_idx = int(n * (1 - test_ratio))
    split_idx = max(1, min(split_idx, n - 1))
    train_idx = np.arange(0, split_idx)
    test_idx = np.arange(split_idx, n)
    return train_idx, test_idx


# ============================================================
# Dataset / DataLoader
# ============================================================

class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


def create_train_val_loaders(
    X_train: np.ndarray,
    y_train: np.ndarray,
    batch_size: int = 32,
    val_ratio: float = VAL_RATIO_WITHIN_TRAIN
):
    n = len(X_train)
    if n < 10:
        raise ValueError(f"训练样本过少，无法稳定划分 train/val，当前 n={n}")

    val_start = int(n * (1 - val_ratio))
    val_start = max(1, min(val_start, n - 1))

    X_tr, X_val = X_train[:val_start], X_train[val_start:]
    y_tr, y_val = y_train[:val_start], y_train[val_start:]

    train_ds = SequenceDataset(X_tr, y_tr)
    val_ds = SequenceDataset(X_val, y_val)

    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin_memory
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin_memory
    )
    return train_loader, val_loader


def create_test_loader(X_test: np.ndarray, y_test: np.ndarray, batch_size: int = 32):
    test_ds = SequenceDataset(X_test, y_test)
    pin_memory = torch.cuda.is_available()
    return DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin_memory
    )


# ============================================================
# Models
# ============================================================

class FeatureAttention(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        inner = max(input_dim // 2, 8)
        self.gate = nn.Sequential(
            nn.Linear(input_dim, inner),
            nn.ReLU(),
            nn.Linear(inner, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor):
        weights = self.gate(x)
        return x * weights, weights


class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, h_seq: torch.Tensor):
        scores = self.score(h_seq).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(h_seq * weights.unsqueeze(-1), dim=1)
        return context, weights


class BaseLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        out_dim: int = 1
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        pred = self.fc(self.dropout(last))
        return pred


class FeatureAttentionLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        out_dim: int = 1
    ):
        super().__init__()
        self.feature_attn = FeatureAttention(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        z, feat_weights = self.feature_attn(x)
        out, _ = self.lstm(z)
        last = out[:, -1, :]
        pred = self.fc(self.dropout(last))
        return pred, feat_weights.mean(dim=1)


class TemporalAttentionLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        out_dim: int = 1
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.temporal_attn = TemporalAttention(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        out, _ = self.lstm(x)
        context, time_weights = self.temporal_attn(out)
        pred = self.fc(self.dropout(context))
        return pred, time_weights


class DualAttentionLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 1,
        dropout: float = 0.2,
        out_dim: int = 1
    ):
        super().__init__()
        self.feature_attn = FeatureAttention(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.temporal_attn = TemporalAttention(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor):
        z, feat_weights = self.feature_attn(x)
        out, _ = self.lstm(z)
        context, time_weights = self.temporal_attn(out)
        pred = self.fc(self.dropout(context))
        aux = {
            "feature_weights": feat_weights.mean(dim=1),
            "time_weights": time_weights,
        }
        return pred, aux


# ============================================================
# Training config
# ============================================================

@dataclass
class TrainConfig:
    hidden_dim: int = 64
    num_layers: int = 1
    dropout: float = 0.2
    learning_rate: float = 1e-3
    batch_size: int = 32
    max_epochs: int = 100
    patience: int = 12
    weight_decay: float = 1e-4


def get_model_search_space(model_name: str) -> List[TrainConfig]:
    return [
        TrainConfig(hidden_dim=64, num_layers=1, dropout=0.2, learning_rate=1e-3, batch_size=32),
        TrainConfig(hidden_dim=96, num_layers=1, dropout=0.2, learning_rate=5e-4, batch_size=32),
        TrainConfig(hidden_dim=96, num_layers=2, dropout=0.3, learning_rate=5e-4, batch_size=32),
    ]


def build_model(
    model_name: str,
    input_dim: int,
    cfg: TrainConfig,
    out_dim: int = 1
) -> nn.Module:
    if model_name == "BaseLSTM":
        return BaseLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "FeatureAttention_LSTM":
        return FeatureAttentionLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "TemporalAttention_LSTM":
        return TemporalAttentionLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    if model_name == "DualAttention_LSTM":
        return DualAttentionLSTM(input_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout, out_dim=out_dim).to(DEVICE)
    raise ValueError(f"Unknown model name: {model_name}")


def build_optimizer_scheduler_criterion(model: nn.Module, cfg: TrainConfig):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=4
    )
    criterion = nn.HuberLoss(delta=1.0)
    return optimizer, scheduler, criterion


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Any = None
):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_n = 0
    preds, labels = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        out = model(X_batch)
        pred = get_prediction(out)
        loss = criterion(pred, y_batch)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item() * len(y_batch)
        total_n += len(y_batch)

        preds.append(pred.detach().cpu().numpy())
        labels.append(y_batch.detach().cpu().numpy())

    avg_loss = total_loss / max(total_n, 1)
    preds = np.concatenate(preds, axis=0)
    labels = np.concatenate(labels, axis=0)
    return avg_loss, preds, labels


def _epoch_reaching_threshold(val_losses: List[float], threshold_value: float) -> Optional[int]:
    for i, v in enumerate(val_losses, start=1):
        if v <= threshold_value:
            return i
    return None


def compute_convergence_metrics(history: Dict[str, List[float]]) -> Dict[str, Optional[float]]:
    val_losses = history["val_loss"]
    train_losses = history["train_loss"]
    lrs = history["lr"]

    if len(val_losses) == 0:
        return {
            "initial_val_loss": None,
            "best_val_loss": None,
            "best_epoch": None,
            "stop_epoch": None,
            "epoch_90": None,
            "epoch_95": None,
            "epochs_to_best_ratio": None,
            "loss_reduction_ratio": None,
            "final_train_loss": None,
            "final_lr": None,
        }

    initial_val = float(val_losses[0])
    best_val = float(min(val_losses))
    best_epoch = int(np.argmin(val_losses) + 1)
    stop_epoch = int(len(val_losses))
    total_gain = max(initial_val - best_val, 0.0)

    if total_gain <= 1e-12:
        epoch_90 = None
        epoch_95 = None
        loss_reduction_ratio = 0.0
    else:
        threshold_90 = initial_val - 0.90 * total_gain
        threshold_95 = initial_val - 0.95 * total_gain
        epoch_90 = _epoch_reaching_threshold(val_losses, threshold_90)
        epoch_95 = _epoch_reaching_threshold(val_losses, threshold_95)
        loss_reduction_ratio = float(total_gain / (abs(initial_val) + 1e-12))

    return {
        "initial_val_loss": initial_val,
        "best_val_loss": best_val,
        "best_epoch": best_epoch,
        "stop_epoch": stop_epoch,
        "epoch_90": epoch_90,
        "epoch_95": epoch_95,
        "epochs_to_best_ratio": float(best_epoch / stop_epoch) if stop_epoch > 0 else None,
        "loss_reduction_ratio": loss_reduction_ratio,
        "final_train_loss": float(train_losses[-1]) if len(train_losses) > 0 else None,
        "final_lr": float(lrs[-1]) if len(lrs) > 0 else None,
    }


def train_with_early_stopping(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: TrainConfig,
    verbose: bool = True
):
    optimizer, scheduler, criterion = build_optimizer_scheduler_criterion(model, cfg)

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "lr": []}

    for epoch in range(cfg.max_epochs):
        train_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss, _, _ = run_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and (epoch < 5 or (epoch + 1) % 10 == 0):
            print(
                f"    Epoch [{epoch+1:03d}/{cfg.max_epochs}] "
                f"train={train_loss:.6f} val={val_loss:.6f} lr={current_lr:.6g}"
            )

        if epochs_no_improve >= cfg.patience:
            if verbose:
                print(f"    提前停止于第 {epoch + 1} 轮")
            break

    if best_state is None:
        raise RuntimeError("Training failed: no best state.")

    convergence_metrics = compute_convergence_metrics(history)

    model.load_state_dict(best_state)
    return model, history, float(best_val_loss), convergence_metrics


def tune_model(
    model_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    input_dim: int,
    out_dim: int,
    verbose: bool = True
):
    configs = get_model_search_space(model_name)
    best_result = None

    if verbose:
        print("\n" + "=" * 72)
        print(f"调参 {model_name} (out_dim={out_dim})")
        print("=" * 72)

    for i, cfg in enumerate(configs, 1):
        if verbose:
            print(f"\n  试验 {i}/{len(configs)} -> {cfg}")

        model = build_model(model_name, input_dim, cfg, out_dim=out_dim)
        train_loader, val_loader = create_train_val_loaders(
            X_train, y_train, batch_size=cfg.batch_size, val_ratio=VAL_RATIO_WITHIN_TRAIN
        )

        model, history, best_val_loss, convergence_metrics = train_with_early_stopping(
            model, train_loader, val_loader, cfg, verbose=verbose
        )

        if verbose:
            print(
                "    收敛统计 -> "
                f"best_epoch={convergence_metrics['best_epoch']}, "
                f"stop_epoch={convergence_metrics['stop_epoch']}, "
                f"epoch_90={convergence_metrics['epoch_90']}, "
                f"epoch_95={convergence_metrics['epoch_95']}"
            )

        result = {
            "config": cfg,
            "history": history,
            "best_val_loss": best_val_loss,
            "state_dict": copy.deepcopy(model.state_dict()),
            "convergence_metrics": convergence_metrics,
        }

        if best_result is None or best_val_loss < best_result["best_val_loss"]:
            best_result = result

    return best_result


# ============================================================
# Scaling helpers
# ============================================================

def fit_transform_X(X_train: np.ndarray, X_test: np.ndarray):
    n_features = X_train.shape[-1]
    scaler_X = StandardScaler()
    scaler_X.fit(X_train.reshape(-1, n_features))
    X_train_scaled = scaler_X.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test_scaled = scaler_X.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)
    return X_train_scaled, X_test_scaled, scaler_X


def fit_transform_Y_multi(Y_train: np.ndarray, Y_test: np.ndarray):
    scaler_Y = StandardScaler()
    scaler_Y.fit(Y_train)
    Y_train_scaled = scaler_Y.transform(Y_train)
    Y_test_scaled = scaler_Y.transform(Y_test)
    return Y_train_scaled, Y_test_scaled, scaler_Y


# ============================================================
# Evaluation helpers
# ============================================================

def evaluate_direct_multi(
    model: nn.Module,
    X_test: np.ndarray,
    Y_test_scaled: np.ndarray,
    Y_test_raw: np.ndarray,
    scaler_Y: StandardScaler,
    batch_size: int
):
    loader = create_test_loader(X_test, Y_test_scaled, batch_size=batch_size)
    criterion = nn.HuberLoss(delta=1.0)

    _, preds_scaled, _ = run_epoch(model, loader, criterion, optimizer=None)
    preds_raw = scaler_Y.inverse_transform(preds_scaled)

    horizon_metrics = {}
    for h in range(FORECAST_HORIZONS):
        horizon_metrics[f"h{h+1}"] = regression_metrics(Y_test_raw[:, h], preds_raw[:, h])

    overall_metrics = regression_metrics(Y_test_raw.reshape(-1), preds_raw.reshape(-1))
    return preds_raw, horizon_metrics, overall_metrics


# ============================================================
# Plotting
# ============================================================

def plot_training_curve(history: Dict[str, List[float]], title: str, save_path: Path):
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="训练集损失")
    plt.plot(history["val_loss"], label="验证集损失")
    plt.title(title)
    plt.xlabel("训练轮次")
    plt.ylabel("Huber损失")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.close()


def plot_three_horizon_prediction(
    target_dates: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title_prefix: str,
    save_path: Path
):
    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)

    for h in range(FORECAST_HORIZONS):
        dt = pd.to_datetime(target_dates[:, h])
        axes[h].plot(dt, y_true[:, h], label="真实值", linewidth=1.8)
        axes[h].plot(dt, y_pred[:, h], label="预测值", linewidth=1.6)
        axes[h].set_title(f"{title_prefix} - 第 t+{h+1} 天预测")
        axes[h].set_ylabel("收盘价")
        axes[h].grid(alpha=0.3)
        axes[h].legend()

    axes[-1].set_xlabel("日期")
    plt.tight_layout()
    plt.savefig(save_path, dpi=180)
    plt.close()


def plot_paper_arch_comparison(stock_dir: Path, stock_label: str, arch_names: List[str]):
    pred_files = {}
    for arch in arch_names:
        f = stock_dir / f"direct_multi_{arch}_predictions.csv"
        if f.exists():
            pred_files[arch] = pd.read_csv(f)

    if len(pred_files) < 2:
        return

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=False)
    first_df = next(iter(pred_files.values()))

    for h in range(1, 4):
        date_col = f"date_t{h}"
        true_col = f"true_t{h}"
        pred_col = f"pred_t{h}"

        dt = pd.to_datetime(first_df[date_col])
        axes[h - 1].plot(dt, first_df[true_col], label="真实值", linewidth=1.9)
        for arch, df in pred_files.items():
            axes[h - 1].plot(dt, df[pred_col], label=arch, linewidth=1.4)
        axes[h - 1].set_title(f"{stock_label} - direct_multi - 第 t+{h} 天预测对比")
        axes[h - 1].grid(alpha=0.3)
        axes[h - 1].legend(fontsize=8)

    axes[-1].set_xlabel("日期")
    plt.tight_layout()
    plt.savefig(stock_dir / "direct_multi_paper_arch_comparison.png", dpi=180)
    plt.close()


def plot_summary_bars(results_df: pd.DataFrame, save_dir: Path):
    summary = results_df.groupby(["arch_model"]).agg(
        mean_overall_R2=("overall_R2", "mean"),
        mean_overall_RMSE=("overall_RMSE", "mean"),
        mean_h1_R2=("h1_R2", "mean"),
        mean_h2_R2=("h2_R2", "mean"),
        mean_h3_R2=("h3_R2", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_epoch_95=("epoch_95", "mean"),
        mean_stop_epoch=("stop_epoch", "mean"),
    ).reset_index()

    labels = summary["arch_model"]

    plt.figure(figsize=(10, 5))
    plt.bar(labels, summary["mean_overall_R2"])
    plt.title("不同模型的平均整体R²")
    plt.ylabel("平均整体R²")
    plt.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(save_dir / "mean_overall_r2_bar.png", dpi=180)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.bar(labels, summary["mean_overall_RMSE"])
    plt.title("不同模型的平均整体RMSE")
    plt.ylabel("平均整体RMSE")
    plt.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(save_dir / "mean_overall_rmse_bar.png", dpi=180)
    plt.close()

    plt.figure(figsize=(10, 5))
    x = np.array([1, 2, 3])
    for _, row in summary.iterrows():
        y = [row["mean_h1_R2"], row["mean_h2_R2"], row["mean_h3_R2"]]
        label = row["arch_model"]
        plt.plot(x, y, marker="o", label=label)
    plt.xticks([1, 2, 3], ["t+1", "t+2", "t+3"])
    plt.title("不同预测步长下的平均R²")
    plt.ylabel("R²")
    plt.xlabel("预测步长")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_dir / "mean_horizon_r2_line.png", dpi=180)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.bar(labels, summary["mean_best_epoch"])
    plt.title("不同模型的平均 best_epoch（越小越快）")
    plt.ylabel("平均 best_epoch")
    plt.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(save_dir / "mean_best_epoch_bar.png", dpi=180)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.bar(labels, summary["mean_epoch_95"])
    plt.title("不同模型的平均 epoch_95（越小越快）")
    plt.ylabel("平均 epoch_95")
    plt.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(save_dir / "mean_epoch_95_bar.png", dpi=180)
    plt.close()


def plot_r2_heatmap(results_df: pd.DataFrame, save_dir: Path):
    pivot_df = results_df.pivot_table(
        index="name",
        columns="arch_model",
        values="overall_R2"
    )

    if pivot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(10, max(8, len(pivot_df) * 0.35)))
    im = ax.imshow(pivot_df.values, aspect="auto")

    ax.set_xticks(np.arange(len(pivot_df.columns)))
    ax.set_xticklabels(list(pivot_df.columns), rotation=25, ha="right", fontsize=9)

    ax.set_yticks(np.arange(len(pivot_df.index)))
    ax.set_yticklabels(pivot_df.index)

    ax.set_title("不同股票与模型的整体R²热力图")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("整体R²")

    for i in range(pivot_df.shape[0]):
        for j in range(pivot_df.shape[1]):
            val = pivot_df.iloc[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)

    plt.tight_layout()
    plt.savefig(save_dir / "overall_r2_heatmap.png", dpi=180)
    plt.close()


# ============================================================
# Result formatting
# ============================================================

def prediction_matrix_to_df(target_dates: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame({
        "date_t1": pd.to_datetime(target_dates[:, 0]),
        "true_t1": y_true[:, 0],
        "pred_t1": y_pred[:, 0],
        "date_t2": pd.to_datetime(target_dates[:, 1]),
        "true_t2": y_true[:, 1],
        "pred_t2": y_pred[:, 1],
        "date_t3": pd.to_datetime(target_dates[:, 2]),
        "true_t3": y_true[:, 2],
        "pred_t3": y_pred[:, 2],
    })


def metrics_row(
    symbol: str,
    name: str,
    sector: str,
    arch_model: str,
    params: int,
    n_rows_merged: int,
    n_samples: int,
    n_train: int,
    n_test: int,
    feature_dim: int,
    horizon_metrics: Dict[str, Dict[str, float]],
    overall_metrics: Dict[str, float],
    best_config: Dict[str, Any],
    convergence_metrics: Dict[str, Any]
):
    row = {
        "symbol": symbol,
        "name": name,
        "sector": sector,
        "experiment_mode": "direct_multi_stock_only",
        "arch_model": arch_model,
        "params": params,
        "merged_rows": n_rows_merged,
        "feature_dim": feature_dim,
        "samples_total": n_samples,
        "train_samples": n_train,
        "test_samples": n_test,
        "overall_MSE": overall_metrics["MSE"],
        "overall_RMSE": overall_metrics["RMSE"],
        "overall_MAE": overall_metrics["MAE"],
        "overall_R2": overall_metrics["R2"],
        "h1_MSE": horizon_metrics["h1"]["MSE"],
        "h1_RMSE": horizon_metrics["h1"]["RMSE"],
        "h1_MAE": horizon_metrics["h1"]["MAE"],
        "h1_R2": horizon_metrics["h1"]["R2"],
        "h2_MSE": horizon_metrics["h2"]["MSE"],
        "h2_RMSE": horizon_metrics["h2"]["RMSE"],
        "h2_MAE": horizon_metrics["h2"]["MAE"],
        "h2_R2": horizon_metrics["h2"]["R2"],
        "h3_MSE": horizon_metrics["h3"]["MSE"],
        "h3_RMSE": horizon_metrics["h3"]["RMSE"],
        "h3_MAE": horizon_metrics["h3"]["MAE"],
        "h3_R2": horizon_metrics["h3"]["R2"],
        "initial_val_loss": convergence_metrics["initial_val_loss"],
        "best_val_loss": convergence_metrics["best_val_loss"],
        "best_epoch": convergence_metrics["best_epoch"],
        "stop_epoch": convergence_metrics["stop_epoch"],
        "epoch_90": convergence_metrics["epoch_90"],
        "epoch_95": convergence_metrics["epoch_95"],
        "epochs_to_best_ratio": convergence_metrics["epochs_to_best_ratio"],
        "loss_reduction_ratio": convergence_metrics["loss_reduction_ratio"],
        "final_train_loss": convergence_metrics["final_train_loss"],
        "final_lr": convergence_metrics["final_lr"],
        "best_config": json.dumps(best_config, ensure_ascii=False),
    }
    return row


# ============================================================
# Single stock experiment
# ============================================================

def run_single_stock_experiment(
    stock_info: Dict[str, str],
    start_date: str = START_DATE,
    end_date: str = END_DATE,
    lookback: int = LOOKBACK,
    verbose: bool = True
):
    symbol = stock_info["symbol"]
    name = stock_info["name"]
    sector = stock_info["sector"]
    stock_label = f"{symbol}_{name}"

    stock_dir = ROOT_DIR / stock_label
    safe_mkdir(stock_dir)

    if verbose:
        print("\n" + "#" * 96)
        print(f"处理股票: {stock_label} | 仅使用个股自身特征")
        print("#" * 96)

    feature_df = prepare_feature_df(stock_info, start_date, end_date)
    feature_df.to_csv(stock_dir / "merged_feature_table.csv", index=False, encoding="utf-8-sig")

    if verbose:
        print(f"  有效行数: {len(feature_df)}")
        print(f"  日期范围: {feature_df['trade_date'].min().date()} -> {feature_df['trade_date'].max().date()}")

    if len(feature_df) < MIN_EFFECTIVE_ROWS:
        raise ValueError(f"{stock_label}: 有效行数过少 ({len(feature_df)})")

    X_all, Y_all, target_dates_all, _, feature_cols = build_multi_horizon_dataset(
        feature_df, lookback=lookback, horizons=FORECAST_HORIZONS
    )

    n_samples = len(X_all)
    if n_samples < 20:
        raise ValueError(f"{stock_label}: 样本数过少 ({n_samples})，无法稳定训练")

    train_idx, test_idx = chronological_split_indices(n_samples, TEST_RATIO)

    X_train_raw, X_test_raw = X_all[train_idx], X_all[test_idx]
    Y_train_raw, Y_test_raw = Y_all[train_idx], Y_all[test_idx]
    dates_test = target_dates_all[test_idx]

    if len(X_test_raw) == 0:
        raise ValueError(f"{stock_label}: 测试集为空")

    X_train_scaled, X_test_scaled, scaler_X = fit_transform_X(X_train_raw, X_test_raw)
    Y_train_scaled_multi, Y_test_scaled_multi, scaler_Y_multi = fit_transform_Y_multi(Y_train_raw, Y_test_raw)

    joblib.dump(scaler_X, stock_dir / "scaler_X.pkl")
    joblib.dump(scaler_Y_multi, stock_dir / "scaler_Y_multi.pkl")
    pd.Series(feature_cols, name="feature").to_csv(stock_dir / "feature_columns.csv", index=False)

    input_dim = X_train_scaled.shape[-1]
    all_rows = []

    for arch in ARCH_NAMES:
        if verbose:
            print(f"\n[{stock_label}] direct_multi | {arch}")

        best_result = tune_model(
            model_name=arch,
            X_train=X_train_scaled,
            y_train=Y_train_scaled_multi,
            input_dim=input_dim,
            out_dim=FORECAST_HORIZONS,
            verbose=verbose
        )

        best_cfg = best_result["config"]
        model = build_model(arch, input_dim, best_cfg, out_dim=FORECAST_HORIZONS)
        model.load_state_dict(best_result["state_dict"])

        preds_raw, horizon_metrics, overall_metrics = evaluate_direct_multi(
            model=model,
            X_test=X_test_scaled,
            Y_test_scaled=Y_test_scaled_multi,
            Y_test_raw=Y_test_raw,
            scaler_Y=scaler_Y_multi,
            batch_size=best_cfg.batch_size
        )

        params = count_parameters(model)

        prediction_matrix_to_df(dates_test, Y_test_raw, preds_raw).to_csv(
            stock_dir / f"direct_multi_{arch}_predictions.csv", index=False
        )

        torch.save(model.state_dict(), stock_dir / f"direct_multi_{arch}_best_model.pth")
        plot_training_curve(
            best_result["history"],
            f"{stock_label} - direct_multi - {arch} 训练过程",
            stock_dir / f"direct_multi_{arch}_training_curve.png"
        )
        plot_three_horizon_prediction(
            dates_test, Y_test_raw, preds_raw,
            f"{stock_label} - direct_multi - {arch}",
            stock_dir / f"direct_multi_{arch}_prediction.png"
        )

        all_rows.append(metrics_row(
            symbol=symbol,
            name=name,
            sector=sector,
            arch_model=arch,
            params=params,
            n_rows_merged=len(feature_df),
            n_samples=n_samples,
            n_train=len(train_idx),
            n_test=len(test_idx),
            feature_dim=input_dim,
            horizon_metrics=horizon_metrics,
            overall_metrics=overall_metrics,
            best_config=asdict(best_cfg),
            convergence_metrics=best_result["convergence_metrics"],
        ))

    plot_paper_arch_comparison(stock_dir, stock_label, ARCH_NAMES)
    stock_result_df = pd.DataFrame(all_rows)
    stock_result_df.to_csv(stock_dir / "stock_all_results.csv", index=False)
    return stock_result_df


# ============================================================
# Global summary
# ============================================================

def summarize_all_results(all_result_df: pd.DataFrame):
    all_result_df.to_csv(ROOT_DIR / "all_stock_all_results.csv", index=False)

    summary = all_result_df.groupby(["arch_model"]).agg(
        stocks=("symbol", "count"),
        mean_overall_R2=("overall_R2", "mean"),
        std_overall_R2=("overall_R2", "std"),
        mean_overall_RMSE=("overall_RMSE", "mean"),
        std_overall_RMSE=("overall_RMSE", "std"),
        mean_h1_R2=("h1_R2", "mean"),
        mean_h2_R2=("h2_R2", "mean"),
        mean_h3_R2=("h3_R2", "mean"),
        mean_params=("params", "mean"),
        mean_feature_dim=("feature_dim", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_epoch_90=("epoch_90", "mean"),
        mean_epoch_95=("epoch_95", "mean"),
        mean_stop_epoch=("stop_epoch", "mean"),
        mean_epochs_to_best_ratio=("epochs_to_best_ratio", "mean"),
    ).reset_index()

    summary = summary.sort_values(["mean_overall_R2"], ascending=[False])
    summary.to_csv(ROOT_DIR / "summary_by_arch.csv", index=False)

    plot_summary_bars(all_result_df, ROOT_DIR)
    plot_r2_heatmap(all_result_df, ROOT_DIR)

    print("\n" + "=" * 170)
    print("股票总体汇总（仅个股自身特征，含收敛速度）")
    print("=" * 170)
    print(
        f"{'arch_model':<28}{'mean_overall_R2':<18}{'mean_overall_RMSE':<20}"
        f"{'mean_best_epoch':<18}{'mean_epoch_95':<16}{'mean_stop_epoch':<18}"
    )
    print("=" * 170)
    for _, row in summary.iterrows():
        print(
            f"{row['arch_model']:<28}"
            f"{row['mean_overall_R2']:<18.4f}"
            f"{row['mean_overall_RMSE']:<20.6f}"
            f"{row['mean_best_epoch']:<18.2f}"
            f"{row['mean_epoch_95']:<16.2f}"
            f"{row['mean_stop_epoch']:<18.2f}"
        )
    print("=" * 170)


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 110)
    print("A股股票 PyTorch GPU 实验：4个LSTM + 仅个股自身25特征 + direct_multi")
    print("=" * 110)
    print(f"Using device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
        print(f"Initial allocated memory: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

    print(f"Stock count: {len(STOCK_POOL)}")
    print(f"Architectures: {ARCH_NAMES}")
    print(f"Lookback: {LOOKBACK}")
    print(f"Horizons: [t+1, t+2, t+3]")
    print(f"Date range: {START_DATE} -> {END_DATE}")
    print(f"Feature count target: {len(STOCK_CORE_FEATURES)}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    pd.DataFrame(STOCK_POOL).to_csv(ROOT_DIR / "stock_pool.csv", index=False)

    check_dependencies()

    all_result_list = []

    for i, stock_info in enumerate(STOCK_POOL, 1):
        try:
            print(f"\n[Main] 进度 {i}/{len(STOCK_POOL)}")
            random_sleep(1.0, 2.0, verbose=True)

            stock_df = run_single_stock_experiment(
                stock_info=stock_info,
                start_date=START_DATE,
                end_date=END_DATE,
                lookback=LOOKBACK,
                verbose=True
            )
            all_result_list.append(stock_df)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"[Error] Failed on stock {stock_info['symbol']} - {stock_info['name']}: {e}")

    if not all_result_list:
        print("[Error] All stock experiments failed.")
        return

    all_result_df = pd.concat(all_result_list, ignore_index=True)
    summarize_all_results(all_result_df)

    paper_stock = next((s for s in STOCK_POOL if s["symbol"] == PAPER_STOCK_SYMBOL), None)
    if paper_stock is not None:
        stock_label = f"{paper_stock['symbol']}_{paper_stock['name']}"
        stock_dir = ROOT_DIR / stock_label
        fig_path = stock_dir / "direct_multi_paper_arch_comparison.png"
        if fig_path.exists():
            print(f"[Info] 论文对比图已保存: {fig_path}")

    print(f"\n全部结果已保存到: {ROOT_DIR.resolve()}")


if __name__ == "__main__":
    main()